In [5]:
import enterprise
from enterprise.pulsar import Pulsar
import enterprise.signals.parameter as parameter
from enterprise.signals import utils
from enterprise.signals import signal_base
from enterprise.signals import selections
from enterprise.signals.selections import Selection
from enterprise.signals import white_signals
from enterprise.signals import gp_signals
from enterprise.signals import deterministic_signals
import enterprise.constants as const

from enterprise_extensions import deterministic


import QuickCW.QuickCW as QuickCW
from QuickCW.QuickMCMCUtils import ChainParams
#import QuickCW.FastLikelihoodNumba as FastLikelihoodNumba

from astropy import units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib import patheffects
from matplotlib import text
import healpy as hp
from healpy.newvisufunc import projview, newprojplot
import numpy as np


In [9]:
c = SkyCoord(ra = '10h05m13.978s', dec = '-07d43m06.89s')
cos_gwtheta = np.cos(np.pi/2 - c.dec.radian)
gwphi = c.ra.radian

print(cos_gwtheta, gwphi)

-0.1343075461635726 2.6408270024866054


In [11]:
c = SkyCoord(ra = '10h05m13.978s', dec = '-07d43m06.89s')
cos_gwtheta = np.cos(0.6387905062299246)
gwphi = 3.3335788713091694

print(cos_gwtheta, gwphi)

0.8028174751911146 3.3335788713091694


In [12]:
import glob
import os
import libstempo as lt

# Change these to match your directory variables
pardir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/"
timdir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim/"

# Find all .par files
parfiles = sorted(glob.glob(os.path.join(pardir, "*.par")))

if not parfiles:
    print(f"❌ No .par files found in {pardir}")
    exit(1)

print(f"Found {len(parfiles)} .par files. Checking for matching .tim files...\n")

for parfile in parfiles:
    base = os.path.splitext(os.path.basename(parfile))[0]
    timfile = os.path.join(timdir, base + ".tim")

    if not os.path.exists(timfile):
        print(f"❌ MISSING TIM file for {base}")
        continue

    try:
        psr = lt.tempopulsar(parfile=parfile, timfile=timfile)
        ntoas = len(psr.toas())
        if ntoas == 0:
            print(f"⚠ {base}: TIM file present but contains 0 TOAs")
        else:
            print(f"✅ {base}: {ntoas} TOAs found")
    except Exception as e:
        print(f"❌ {base}: Error reading files → {e}")


Found 33 .par files. Checking for matching .tim files...

✅ J0030+0451: 183 TOAs found
✅ J0034-0534: 183 TOAs found
✅ J0218+4232: 183 TOAs found
✅ J0437-4715: 183 TOAs found
✅ J0613-0200: 183 TOAs found
✅ J0621+1002: 183 TOAs found
✅ J0711-6830: 183 TOAs found
✅ J0751+1807: 183 TOAs found
✅ J0900-3144: 183 TOAs found
✅ J1012+5307: 183 TOAs found
✅ J1022+1001: 183 TOAs found
✅ J1024-0719: 183 TOAs found
✅ J1045-4509: 183 TOAs found
✅ J1455-3330: 183 TOAs found
✅ J1600-3053: 183 TOAs found
✅ J1603-7202: 183 TOAs found
✅ J1614-2230: 183 TOAs found
✅ J1640+2224: 183 TOAs found
✅ J1643-1224: 183 TOAs found
✅ J1713+0747: 183 TOAs found
✅ J1730-2304: 183 TOAs found
✅ J1744-1134: 183 TOAs found
✅ J1857+0943: 183 TOAs found
✅ J1909-3744: 183 TOAs found
✅ J1918-0642: 183 TOAs found
✅ J1939+2134: 183 TOAs found
✅ J1944+0907: 183 TOAs found
✅ J2010-1323: 183 TOAs found
✅ J2124-3358: 183 TOAs found
✅ J2129-5721: 183 TOAs found
✅ J2145-0750: 183 TOAs found
✅ J2229+2643: 183 TOAs found
✅ J2317+1439: 

In [13]:
import os
import glob
import libstempo as lt

# Set your directories here
pardir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/"
timdir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim/"

# Find all par files
parfiles = sorted(glob.glob(os.path.join(pardir, "*.par")))

if not parfiles:
    print(f"❌ No .par files found in {pardir}")
    exit(1)

print(f"Found {len(parfiles)} .par files. Checking with Tempo2/libstempo...\n")

fail_count = 0

for par in parfiles:
    name = os.path.splitext(os.path.basename(par))[0]
    tim = os.path.join(timdir, name + ".tim")

    if not os.path.exists(tim):
        print(f"⚠ Missing TIM file for {name}")
        fail_count += 1
        continue

    try:
        psr = lt.tempopulsar(parfile=par, timfile=tim)
        ntoas = len(psr.toas())
        if ntoas == 0:
            print(f"⚠ {name}: Loaded but has 0 TOAs")
            fail_count += 1
        else:
            print(f"✅ {name}: {ntoas} TOAs loaded successfully")
    except Exception as e:
        print(f"❌ {name}: Error loading → {e}")
        fail_count += 1

print(f"\nCheck complete. {fail_count} pulsar(s) failed.")
if fail_count > 0:
    print("⚠ Fix the above issues before running MCMC.")


Found 33 .par files. Checking with Tempo2/libstempo...

✅ J0030+0451: 183 TOAs loaded successfully
✅ J0034-0534: 183 TOAs loaded successfully
✅ J0218+4232: 183 TOAs loaded successfully
✅ J0437-4715: 183 TOAs loaded successfully
✅ J0613-0200: 183 TOAs loaded successfully
✅ J0621+1002: 183 TOAs loaded successfully
✅ J0711-6830: 183 TOAs loaded successfully
✅ J0751+1807: 183 TOAs loaded successfully
✅ J0900-3144: 183 TOAs loaded successfully
✅ J1012+5307: 183 TOAs loaded successfully
✅ J1022+1001: 183 TOAs loaded successfully
✅ J1024-0719: 183 TOAs loaded successfully
✅ J1045-4509: 183 TOAs loaded successfully
✅ J1455-3330: 183 TOAs loaded successfully
✅ J1600-3053: 183 TOAs loaded successfully
✅ J1603-7202: 183 TOAs loaded successfully
✅ J1614-2230: 183 TOAs loaded successfully
✅ J1640+2224: 183 TOAs loaded successfully
✅ J1643-1224: 183 TOAs loaded successfully
✅ J1713+0747: 183 TOAs loaded successfully
✅ J1730-2304: 183 TOAs loaded successfully
✅ J1744-1134: 183 TOAs loaded successfull

In [14]:
import glob
import os
from collections import Counter

pardir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/"

for parfile in sorted(glob.glob(os.path.join(pardir, "*.par"))):
    with open(parfile, "r") as f:
        lines = f.readlines()
    params = [line.split()[0] for line in lines if line.strip() and not line.startswith("#")]
    counts = Counter(params)
    if counts.get("NE_SW", 0) > 1:
        print(f"⚠ {os.path.basename(parfile)}: {counts['NE_SW']} NE_SW entries found")

        

⚠ J0030+0451.par: 2 NE_SW entries found
⚠ J0034-0534.par: 2 NE_SW entries found
⚠ J0218+4232.par: 2 NE_SW entries found
⚠ J0437-4715.par: 2 NE_SW entries found
⚠ J0613-0200.par: 2 NE_SW entries found
⚠ J0621+1002.par: 2 NE_SW entries found
⚠ J0711-6830.par: 2 NE_SW entries found
⚠ J0751+1807.par: 2 NE_SW entries found
⚠ J0900-3144.par: 2 NE_SW entries found
⚠ J1012+5307.par: 2 NE_SW entries found
⚠ J1022+1001.par: 2 NE_SW entries found
⚠ J1024-0719.par: 2 NE_SW entries found
⚠ J1045-4509.par: 2 NE_SW entries found
⚠ J1455-3330.par: 2 NE_SW entries found
⚠ J1600-3053.par: 2 NE_SW entries found
⚠ J1603-7202.par: 2 NE_SW entries found
⚠ J1614-2230.par: 2 NE_SW entries found
⚠ J1640+2224.par: 2 NE_SW entries found
⚠ J1643-1224.par: 2 NE_SW entries found
⚠ J1713+0747.par: 2 NE_SW entries found
⚠ J1730-2304.par: 2 NE_SW entries found
⚠ J1744-1134.par: 2 NE_SW entries found
⚠ J1857+0943.par: 2 NE_SW entries found
⚠ J1909-3744.par: 2 NE_SW entries found
⚠ J1918-0642.par: 2 NE_SW entries found


In [15]:
import libstempo as lt
p = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/J0030+0451.par"
t = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim/J0030+0451.tim"
psr = lt.tempopulsar(parfile=p, timfile=t)
print(len(psr.toas()))

183


In [16]:
'''
import glob
import os
import shutil

pardir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/"

for parfile in sorted(glob.glob(os.path.join(pardir, "*.par"))):
    with open(parfile, "r") as f:
        lines = f.readlines()

    seen_ne_sw = False
    cleaned_lines = []
    for line in lines:
        if line.strip().startswith("NE_SW"):
            if not seen_ne_sw:
                cleaned_lines.append(line)
                seen_ne_sw = True
            else:
                print(f"Removing duplicate NE_SW from {os.path.basename(parfile)}: {line.strip()}")
        else:
            cleaned_lines.append(line)

    # Backup original
    shutil.copy(parfile, parfile + ".bak")

    # Write cleaned file
    with open(parfile, "w") as f:
        f.writelines(cleaned_lines)

print("✅ All .par files cleaned. Original files backed up with .bak extension.")
'''

Removing duplicate NE_SW from J0030+0451.par: NE_SW          4.000
Removing duplicate NE_SW from J0034-0534.par: NE_SW          4.000
Removing duplicate NE_SW from J0218+4232.par: NE_SW          4.000
Removing duplicate NE_SW from J0437-4715.par: NE_SW          4.000
Removing duplicate NE_SW from J0613-0200.par: NE_SW          4.000
Removing duplicate NE_SW from J0621+1002.par: NE_SW          4.000
Removing duplicate NE_SW from J0711-6830.par: NE_SW          4.000
Removing duplicate NE_SW from J0751+1807.par: NE_SW          4.000
Removing duplicate NE_SW from J0900-3144.par: NE_SW          4.000
Removing duplicate NE_SW from J1012+5307.par: NE_SW          4.000
Removing duplicate NE_SW from J1022+1001.par: NE_SW          4.000
Removing duplicate NE_SW from J1024-0719.par: NE_SW          4.000
Removing duplicate NE_SW from J1045-4509.par: NE_SW          4.000
Removing duplicate NE_SW from J1455-3330.par: NE_SW          4.000
Removing duplicate NE_SW from J1600-3053.par: NE_SW          4

In [ ]:
import glob
import os

pardir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/"

for parfile in sorted(glob.glob(os.path.join(pardir, "*.par"))):
    with open(parfile, "r") as f:
        lines = f.readlines()
    
    ne_sw_values = [line.strip() for line in lines if line.strip().startswith("NE_SW")]
    
    if ne_sw_values:
        print(f"{os.path.basename(parfile)}:")
        for val in ne_sw_values:
            print(f"   {val}")

In [ ]:
#/hyrule/data/users/wcfiore/MDC2